<a href="https://colab.research.google.com/github/ririkli/hotel-cancellation-dashboard/blob/main/dashboard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%capture

!pip install opendatasets

In [ ]:
!pip install dash plotly pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 73.4 MB/s eta 0:00:00


In [ ]:
import opendatasets as od

od.download("https://www.kaggle.com/datasets/jessemostipak/hotel-booking-demand")

Please provide your Kaggle credentials to download this dataset. Learn more: http://bit.ly/kaggle-creds
Your Kaggle username: ririkli
Your Kaggle Key: ··········
Dataset URL: https://www.kaggle.com/datasets/jessemostipak/hotel-booking-demand


100%|██████████| 1.25M/1.25M [00:00<00:00, 132MB/s]

In [ ]:
import os
import pandas as pd

data_folder = "/content/hotel-booking-demand"
df = pd.read_csv(os.path.join(data_folder, "hotel_bookings.csv"))
df

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.00,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.00,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,No Deposit,NaN,NaN,0,Transient,75.00,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,No Deposit,304.0,NaN,0,Transient,75.00,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,No Deposit,240.0,NaN,0,Transient,98.00,0,1,Check-Out,2015-07-03
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
119385,City Hotel,0,23,2017,August,35,30,2,5,2,...,No Deposit,394.0,NaN,0,Transient,96.14,0,0,Check-Out,2017-09-06
119386,City Hotel,0,102,2017,August,35,31,2,5,3,...,No Deposit,9.0,NaN,0,Transient,225.43,0,2,Check-Out,2017-09-07
119387,City Hotel,0,34,2017,August,35,31,2,5,2,...,No Deposit,9.0,NaN,0,Transient,157.71,0,4,Check-Out,2017-09-07
119388,City Hotel,0,109,2017,August,35,31,2,5,2,...,No Deposit,89.0,NaN,0,Transient,104.40,0,0,Check-Out,2017-09-07


In [ ]:
!pip install dash -q --upgrade

from dash import Dash, html, dcc, callback, Output, Input, dash_table
import pandas as pd
import numpy as np
import plotly.express as px
from google.colab import output
import threading
import time

app = Dash(__name__)

DARK_STYLE = {
    'backgroundColor': '#1e1e1e',
    'color': '#FFFFFF',
    'padding': '20px',
}

app.layout = html.Div(style=DARK_STYLE, children=[
    html.H2('Hotel Cancellation Dashboard', style={'marginBottom': '20px'}),

    html.Div([
        html.Div([
            html.Div([
                html.Label('Year', style={'fontWeight': 'bold', 'display': 'block', 'marginBottom': '5px'}),
                dcc.Dropdown(
                    id='year-dropdown',
                    options=[{'label': str(year), 'value': year} for year in sorted(df['arrival_date_year'].unique())],
                    value=[df['arrival_date_year'].max()],
                    multi=True,
                    style={'color': '#000000'}
                )
            ], style={'marginBottom': '15px'}),

            html.Div([
                html.Label('Hotel', style={'fontWeight': 'bold', 'display': 'block', 'marginBottom': '5px'}),
                dcc.Dropdown(
                    id='hotel-dropdown',
                    options=[{'label': h, 'value': h} for h in sorted(df['hotel'].unique())],
                    value=list(df['hotel'].unique()),
                    multi=True,
                    style={'color': '#000000'}
                )
            ], style={'marginBottom': '25px'}),

            html.Div([
                html.H4('Cancellation Rate', style={'fontSize': '16px', 'color': '#b3b3b3', 'margin': '0'}),
                html.Div(id='cancellation-rate-kpi', style={'fontSize': '64px', 'fontWeight': 'bold', 'color': '#e74c3c', 'marginTop': '10px'})
            ], style={'backgroundColor': '#2d2d2d', 'padding': '20px', 'borderRadius': '4px', 'height': '150px'})
        ], style={'width': '23%', 'display': 'inline-block', 'verticalAlign': 'top'}),

        html.Div([
            html.H4('Cancellation Risk Matrix', style={'margin': '0 0 10px 0', 'fontSize': '16px'}),
            dash_table.DataTable(
                id='risk-matrix-table',
                style_header={
                    'backgroundColor': '#2d2d2d',
                    'color': 'white',
                    'fontWeight': 'bold',
                    'border': '1px solid #444'
                },
                style_data={
                    'backgroundColor': '#2d2d2d',
                    'color': 'white',
                    'border': '1px solid #444'
                },
                style_cell={'padding': '10px', 'textAlign': 'center', 'fontFamily': 'Arial'}
            )
        ], style={'width': '74%', 'display': 'inline-block', 'float': 'right', 'verticalAlign': 'top'})
    ], style={'marginBottom': '20px'}),

    html.Div([
        html.Div([
            html.H4('Cancellation Rate Trend', style={'borderLeft': '4px solid #3498db', 'paddingLeft': '10px', 'margin': '0 0 10px 0'}),
            dcc.Graph(id='cancellation-trend-graph')
        ], style={'width': '65%', 'display': 'inline-block', 'verticalAlign': 'top'}),

        html.Div([
            html.H4('Cancellations by Market Segment', style={'borderLeft': '4px solid #9b59b6', 'paddingLeft': '10px', 'margin': '0 0 10px 0'}),
            dcc.Graph(id='segment-bar-graph')
        ], style={'width': '33%', 'display': 'inline-block', 'float': 'right', 'verticalAlign': 'top'})
    ])
])

@callback(
    [Output('cancellation-rate-kpi', 'children'),
     Output('risk-matrix-table', 'data'),
     Output('risk-matrix-table', 'columns'),
     Output('risk-matrix-table', 'style_data_conditional'),
     Output('cancellation-trend-graph', 'figure'),
     Output('segment-bar-graph', 'figure')],
    [Input('year-dropdown', 'value'),
     Input('hotel-dropdown', 'value')]
)
def update_dashboard(selected_years, selected_hotels):
    if not selected_years:
        selected_years = list(df['arrival_date_year'].unique())
    if not selected_hotels:
        selected_hotels = list(df['hotel'].unique())

    filtered_df = df[(df['arrival_date_year'].isin(selected_years)) & (df['hotel'].isin(selected_hotels))].copy()

    total_cancel_rate = filtered_df['is_canceled'].mean() * 100 if len(filtered_df) > 0 else 0
    kpi_text = f"{total_cancel_rate:.2f}%".replace('.', ',')

    def get_booking_window(lt):
        if lt <= 7: return '1. Within 7 Days'
        elif lt <= 30: return '2. 8-30 Days'
        elif lt <= 90: return '3. 31-90 Days'
        else: return '4. 90+ Days'

    def get_price_category(adr):
        if adr <= 50: return '1. Up to $50 (Economy)'
        elif adr <= 100: return '2. $50 - $100 (Standard)'
        elif adr <= 150: return '3. $100 - $150 (Comfort)'
        else: return '4. Over $150 (Premium)'

    matrix_df = filtered_df.copy()
    matrix_df['Booking window'] = matrix_df['lead_time'].apply(get_booking_window)
    matrix_df['Price Category'] = matrix_df['adr'].apply(get_price_category)

    pivot_df = matrix_df.groupby(['Booking window', 'Price Category'])['is_canceled'].mean().reset_index()
    pivot_df['is_canceled'] = (pivot_df['is_canceled'] * 100).round(2)

    matrix_table = pivot_df.pivot(index='Booking window', columns='Price Category', values='is_canceled').reset_index()

    columns = [{"name": "Booking window", "id": "Booking window"}] + [
        {"name": col, "id": col} for col in matrix_table.columns if col != 'Booking window'
    ]

    table_data = matrix_table.to_dict('records')

    style_conditional = []
    for row in table_data:
        for col in matrix_table.columns:
            if col != 'Booking window':
                val = row.get(col, 0)
                if pd.isna(val): continue
                if val < 15: bg_color = '#2980b9'
                elif val < 35: bg_color = '#8e44ad'
                else: bg_color = '#c0392b'
                style_conditional.append({
                    'if': {'row_index': table_data.index(row), 'column_id': col},
                    'backgroundColor': bg_color,
                    'color': 'white'
                })

    months_order = ['January', 'February', 'March', 'April', 'May', 'June', 'July', 'August', 'September', 'October', 'November', 'December']
    trend_data = filtered_df.groupby(['hotel', 'arrival_date_month'])['is_canceled'].mean().reset_index()
    trend_data['arrival_date_month'] = pd.Categorical(trend_data['arrival_date_month'], categories=months_order, ordered=True)
    trend_data = trend_data.sort_values(['hotel', 'arrival_date_month'])

    fig_trend = px.line(
        trend_data,
        x='arrival_date_month',
        y='is_canceled',
        color='hotel',
        facet_row='hotel',
        labels={'is_canceled': 'Cancellation Rate', 'arrival_date_month': 'Month'},
        markers=True
    )
    fig_trend.update_layout(
        template='plotly_dark',
        paper_bgcolor='#1e1e1e',
        plot_bgcolor='#2d2d2d',
        height=450,
        margin=dict(l=40, r=20, t=10, b=20),
        showlegend=False
    )
    fig_trend.update_yaxes(tickformat=".2f", matches=None)
    fig_trend.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))

    segment_data = filtered_df.groupby('market_segment').agg(
        volume=('is_canceled', 'count'),
        cancel_rate=('is_canceled', 'mean')
    ).reset_index()
    segment_data['cancel_rate'] = segment_data['cancel_rate'] * 100
    segment_data = segment_data.sort_values(by='volume', ascending=False)

    fig_bar = px.bar(
        segment_data,
        x='market_segment',
        y='volume',
        text=segment_data['cancel_rate'].map('{:,.2f}%'.format),
        labels={'volume': 'Volume', 'market_segment': 'Segment'}
    )
    fig_bar.update_layout(
        template='plotly_dark',
        paper_bgcolor='#1e1e1e',
        plot_bgcolor='#2d2d2d',
        height=450,
        margin=dict(l=40, r=20, t=10, b=40),
        showlegend=False
    )
    fig_bar.update_traces(
        textposition='outside',
    )

    return kpi_text, table_data, columns, style_conditional, fig_trend, fig_bar

try:
    output.clear()
except:
    pass

import random
port = random.randint(8000, 8999)

def run_app():
    app.run(debug=False, port=port)

threading.Thread(target=run_app, daemon=True).start()
time.sleep(3)

output.serve_kernel_port_as_window(port)

<IPython.core.display.Javascript object>

Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>